In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
catalog = "cine_data_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

## 1.) dim_movies

In [0]:
df_silver_filmes = spark.table(f"{silver_schema}.tb_info_filmes")

df_dim_movies = (
    df_silver_filmes
    .withColumn("sk_movie_id", F.row_number().over(Window.orderBy("id_filme")))
    .select("sk_movie_id", "id_filme", "titulo",
            "data_lancamento", "ano_lancamento", "duracao_minutos", "idioma_original", "status_filme", "sinopse")
)

df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_movies")

print(f"dim_movies: {df_dim_movies.count()} movies")
display(df_dim_movies.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_movies: 97879 movies


sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
1,14564,Rings,2017-02-01T00:00:00.000Z,2017,102,en,Lançado,"\Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die."
2,32471,Mixtape,2021-12-03T00:00:00.000Z,2021,94,en,Lançado,null
3,38258,Grizzly II: Revenge,2020-02-17T00:00:00.000Z,2020,74,en,Lançado,\All hell breaks loose when a giant grizzly
4,38492,Billy Joel - Live at Yankee Stadium,2022-06-22T00:00:00.000Z,2022,86,en,Lançado,Billy Joel plays his greatest hits in the Big Apple.
5,38700,Bad Boys for Life,2020-01-15T00:00:00.000Z,2020,124,en,Lançado,"Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel."
6,42018,The Horse Thief,2019-03-19T00:00:00.000Z,2019,88,zh,Lançado,"Devout Buddhists, Norbu and Dolma live with their young son Tashi in a clan in Tibet. Norbu is a highwayman. After Norbu is charged with stealing from the temple, he and his family are banished. Impoverished and marginalized, they can do little when their beloved son becomes ill. Tashi dies of a fever. After a second son is born, Norbu focuses his every action on keeping this child alive, seeking re-admission to the clan for his wife and child, then risking all to save them from isolation and starvation in winter."
7,42330,Monkey Magic,2018-09-22T00:00:00.000Z,2018,66,zh,Lançado,"Dearth Voyd, the ruler of the dark side of the universe, wants to turn the human world into a place of violence and evil!! However, a huge meteor falls upon Flower-Fruit mountain giving birth to Kongo, a monkey made of stone! Kongo, now sets out on his quest to fight evil and restore order to a desperate land. Episodes 1-3"
8,43074,Ghostbusters,2016-07-14T00:00:00.000Z,2016,117,en,Lançado,"Following a ghost invasion of Manhattan, paranormal enthusiasts Erin Gilbert and Abby Yates, nuclear engineer Jillian Holtzmann, and subway worker Patty Tolan band together to stop the otherworldly threat."
9,45033,20 Seconds of Joy,2018-01-01T00:00:00.000Z,2018,60,de,Lançado,"Traces the story of an extreme athlete, past and present; but also explores the psychology behind life, death, risk and the confrontation of fear."
10,46983,The Song of Styrene,null,null,13,fr,Lançado,Le chant du Styrène is a 1958 French documentary film directed by Alain Resnais. The film was an order by French industrial group Pechiney to highlight the merits of plastics.


### 2.) dim_genres

In [0]:
df_silver_genres = spark.table(f"{silver_schema}.tb_generos")

df_dim_genres = (
    df_silver_genres
    .withColumn("sk_genre_id", F.row_number().over(Window.orderBy("genero")))
    .withColumnRenamed("genero", "nome_genero")
    .select("sk_genre_id", "nome_genero")
)

df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_genres")

print(f"dim_movies: {df_dim_genres.count()} genres")
display(df_dim_genres.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_movies: 19 genres


sk_genre_id,nome_genero
1,Action
2,Adventure
3,Animation
4,Comedy
5,Crime
6,Documentary
7,Drama
8,Family
9,Fantasy
10,History


###  3.) dim_people

In [0]:
df_silver_people = spark.table(f"{silver_schema}.tb_pessoas_empresas")

df_dim_people = (
    df_silver_people
    .withColumn("sk_person_id", F.row_number().over(Window.orderBy("nome")))
    .withColumnRenamed("nome", "nome_pessoa")
    .withColumnRenamed("tipo_entidade", "tipo_pessoa")
    .filter(F.col("tipo_pessoa").isin("Ator", "Diretor", "Roteirista"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

df_dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_people")

print(f"dim_people: {df_dim_people.count()} people")
display(df_dim_people.limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_people: 420448 people


sk_person_id,nome_pessoa,tipo_pessoa
3,"""1969 (\""""careful With That Axe""""""",Ator
14,"""\francis """"\""""indio""""\"""" Disla Ferreira""""""""""juan Jose Namnun\""""",Diretor
19,"""antonio Morales.\""""",Diretor
20,"""as Documented In \""""radioactive Veteran.\""""\""""\""""""""""",Diretor
39,'ana Ika,Ator


### 4.) dim_companies

In [0]:
df_silver_companies = spark.table(f"{silver_schema}.tb_pessoas_empresas")

df_dim_companies = (
    df_silver_companies
    .withColumn("sk_company_id", F.row_number().over(Window.orderBy("nome")))
    .withColumnRenamed("nome", "nome_produtora")
    .filter(F.col("tipo_entidade").isin("Produtora"))
    .select("sk_company_id", "nome_produtora")
)

df_dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_company")

print(f"dim_companies: {df_dim_companies.count()} companies")
display(df_dim_companies.limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_companies: 44710 companies


sk_company_id,nome_produtora
1,
2,"""14 April 1969 (\""""afternoon\""""\"""""""""""
4,"""\"""" And Many More.\"""""""""""
5,"""\"""" \""""\""""this Is It"""""""
6,"""\"""" \""""\""""where I Come From"""""""


### 5.) dim_reviews

In [0]:
df_silver_reviews = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")

df_dim_reviews = (
    df_silver_reviews

    # 1. Resume as avaliações por filme
    .groupBy("id_filme")
    .agg(
        F.count("nota_usuario").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"),
    )

    # 2. Busca a chave do filme na dim_movies
    .join(df_dim_movies.select("id_filme", "sk_movie_id"), on="id_filme", how="inner")

    # 3. Gera a chave da dimensão
    .withColumn("sk_review_id", F.row_number().over(Window.orderBy("sk_movie_id")).cast("bigint"))

    .select(
        "sk_review_id",
        F.col("sk_movie_id").cast("bigint"),
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios",
    )
)

df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_reviews")

print(f"dim_reviews: {df_dim_reviews.count()} filmes avaliados")
display(df_dim_reviews.limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_reviews: 27303 filmes avaliados


sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
1,2,1,8.3
2,3,2,5.15
3,4,1,2.5
4,5,1,0.1
5,10,1,0.2


### 6.) fact_movies_performance


In [0]:
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")
df_silver_financeiro = spark.table(f"{silver_schema}.tb_financeiro_filmes")
df_silver_filmes = spark.table(f"{silver_schema}.tb_metricas_engajamento")

# 1. Filmes lançados
df_filmes = (
    df_dim_movies
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
)

# 2. Métricas financeiras
df_financeiro = (
    df_silver_financeiro
    .dropDuplicates(["id_filme"])
    .select(
        "id_filme", "orcamento_usd", "receita_usd", "lucro_usd",
        "orcamento_brl", "receita_brl", "lucro_brl",
    )
)

# 3. Métricas de engajamento
df_engajamento = (
    df_silver_filmes
    .dropDuplicates(["id_filme"])
    .select(
        "id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
        "nota_media_imdb", "qtd_votos_imdb",
    )
)

# 4. Junta tudo a partir dos filmes
df_fact = (
    df_filmes
    .join(df_financeiro, on="id_filme", how="left")
    .join(df_engajamento, on="id_filme", how="left")
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int"),
    )
)

df_fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_movies_performance")

# 5. Verificação do grão: as duas contagens devem ser iguais
print(f"Filmes lançados: {df_filmes.count()}")
print(f"Linhas na fato:  {df_fact.count()}")
display(df_fact)

Filmes lançados: 96463
Linhas na fato:  96463


sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1,25000000.00,83080890.00,58080890.00,124822500.00,414814575.68,289992075.68,24.584,4.966,2375,null,46286
2,null,null,null,null,null,null,8.929,7.064,118,6.6,4617
3,7.50,null,null,37.45,null,null,null,3.161,28,null,null
4,null,null,null,null,null,null,2.98,7.1,null,7.8,212
5,90.00,426505244.00,426505154.00,449.36,2129498032.77,2129497583.41,46.619,7.139,7570,6.5,199420
6,null,null,null,null,null,null,3.403,6.63,27,6.8,1750
7,null,null,null,null,null,null,1.561,10.0,1,6.4,34
8,144000000.00,229147509.00,85147509.00,718977600.00,1144110597.69,425132997.69,40.052,5.371,5976,null,260417
9,337200.00,null,null,1683605.88,null,null,0.6,8.0,2,null,154
10,null,null,null,null,null,null,0.693,8.5,2,7.0,1209


### Perguntas de Negócio

####1. Soma das receitas todos os filmes lançados da base

In [0]:
%sql
SELECT SUM(receita_brl) AS receita_total_brl
FROM cine_data_medallion.gold.fact_movies_performance;

receita_total_brl
808186091331.71


#### 2. Os 5 filmes mais populares

In [0]:
%sql
SELECT m.titulo, f.popularidade
FROM cine_data_medallion.gold.fact_movies_performance f
JOIN cine_data_medallion.gold.dim_movies m ON f.sk_movie_id = m.sk_movie_id
ORDER BY f.popularidade DESC
LIMIT 5;

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The Fear Footage 2: Curse of the Tape,2019.0
wwe survivor series 2018,2018.0


#### 4. Top 10 filmes por receita

In [0]:
%sql
SELECT *
FROM (
    SELECT
        m.titulo,
        f.receita_usd,
        f.receita_brl,
        RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao
    FROM cine_data_medallion.gold.fact_movies_performance f
    JOIN cine_data_medallion.gold.dim_movies m ON f.sk_movie_id = m.sk_movie_id
    WHERE f.receita_usd IS NOT NULL
)
WHERE posicao <= 10
ORDER BY posicao;

titulo,receita_usd,receita_brl,posicao
Avengers: Endgame,2800000000.00,13980120000.00,1
Avatar: The Way of Water,2320250281.00,11584777628.00,2
AVENGERS: INFINITY WAR,2052415039.00,10247503048.22,3
spider-man: no way home,1921847111.00,9595590440.51,4
The Lion King,1663075401.00,8303569169.65,5
Top Gun: Maverick,1488732821.00,7433094101.97,6
Barbie,1428545028.00,7132582470.30,7
The Super Mario Bros. Movie,1355725263.00,6769000665.63,8
Black Panther,1349926083.00,6740045939.81,9
Star Wars: The Last Jedi,1332698830.00,6654031988.31,10
